In [ ]:
# ================================================================
# CELL 1: IMPORT + CONFIG + DATASET + MODEL + UTILS
# MINI LM ONLY VERSION
# ================================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate scikit-learn scipy psutil

import os
import sys
import csv
import json
import time
import math
import random
import logging
import platform

from pathlib import Path

import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import (
    DataLoader,
    Dataset
)

from torch.optim import AdamW

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoConfig,
    get_linear_schedule_with_warmup,
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from scipy.stats import (
    pearsonr,
    spearmanr
)

# ================================================================
# OUTPUT
# ================================================================

OUTPUT_ROOT = "/content/drive/MyDrive/MINILM_MTL_RESULTS"

Path(OUTPUT_ROOT).mkdir(
    parents=True,
    exist_ok=True
)

# ================================================================
# HP
# ================================================================

HP = dict(

    learning_rate=2e-5,

    batch_size=16,

    eval_batch_size=32,

    max_seq_length=256,

    weight_decay=0.01,

    warmup_ratio=0.1,

    seed=42,

    grad_clip=1.0,

    hidden_dropout=0.2,

    patience=3,

    max_epochs=5,

    num_workers=2,

    keep_n_ckpts=2,

    bench_max_batches=50,
)

# ================================================================
# DEVICE
# ================================================================

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

USE_AMP = torch.cuda.is_available()

print("DEVICE:", DEVICE)

# ================================================================
# MINI LM ONLY
# ================================================================

MODEL_NAME = "microsoft/MiniLM-L12-H384-uncased"

HIDDEN_SIZE = 384

EXP_NAME = "MiniLM-MTL-FFT"

# ================================================================
# TASKS
# ================================================================

TASKS = {

    "sst2": {

        "hf": ("glue", "sst2"),

        "type": "cls",

        "a": "sentence",

        "b": None,

        "num_labels": 2,
    },

    "qqp": {

        "hf": ("glue", "qqp"),

        "type": "cls",

        "a": "question1",

        "b": "question2",

        "num_labels": 2,
    },

    "stsb": {

        "hf": ("glue", "stsb"),

        "type": "reg",

        "a": "sentence1",

        "b": "sentence2",

        "num_labels": 1,
    },
}

TASK_LIST = [
    "sst2",
    "qqp",
    "stsb"
]

K = len(TASK_LIST)

# ================================================================
# SEED
# ================================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)

set_seed(HP["seed"])

# ================================================================
# LOGGER
# ================================================================

def make_logger(name, log_path):

    Path(log_path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    lg = logging.getLogger(name)

    lg.setLevel(logging.INFO)

    lg.propagate = False

    for h in list(lg.handlers):

        lg.removeHandler(h)

    fmt = logging.Formatter(
        "[%(asctime)s] [%(levelname)s] %(message)s",
        "%Y-%m-%d %H:%M:%S"
    )

    fh = logging.FileHandler(
        log_path,
        mode="a",
        encoding="utf-8"
    )

    fh.setFormatter(fmt)

    sh = logging.StreamHandler(sys.stdout)

    sh.setFormatter(fmt)

    lg.addHandler(fh)

    lg.addHandler(sh)

    return lg

# ================================================================
# HISTORY WRITER
# ================================================================

class HistoryWriter:

    def __init__(self, out_dir):

        self.csv_path = (
            Path(out_dir)
            / "history.csv"
        )

        self.json_path = (
            Path(out_dir)
            / "history.json"
        )

        self.records = []

        self.fields = []

    def append(self, row):

        for k in row:

            if k not in self.fields:

                self.fields.append(k)

        self.records.append(row)

        with open(
            self.csv_path,
            "w",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.DictWriter(
                f,
                fieldnames=self.fields
            )

            writer.writeheader()

            for r in self.records:

                writer.writerow({
                    k: r.get(k, "")
                    for k in self.fields
                })

        with open(
            self.json_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                self.records,
                f,
                indent=2,
                default=str
            )

# ================================================================
# DATASET
# ================================================================

class GlueTaskDS(Dataset):

    def __init__(
        self,
        hf,
        task,
        tok,
        max_len
    ):

        m = TASKS[task]

        self.a = list(hf[m["a"]])

        self.b = (
            list(hf[m["b"]])
            if m["b"]
            else None
        )

        self.y = np.asarray(

            hf["label"],

            dtype=np.float32
            if m["type"] == "reg"
            else np.int64
        )

        self.tok = tok

        self.max_len = max_len

        self.task = task

        self.type = m["type"]

    def __len__(self):

        return len(self.a)

    def __getitem__(self, i):

        if self.b is None:

            enc = self.tok(

                self.a[i],

                truncation=True,

                max_length=self.max_len,

                padding="max_length",

                return_tensors="pt"
            )

        else:

            enc = self.tok(

                self.a[i],

                self.b[i],

                truncation=True,

                max_length=self.max_len,

                padding="max_length",

                return_tensors="pt"
            )

        item = {

            k: v.squeeze(0)

            for k, v in enc.items()
        }

        item["labels"] = torch.tensor(

            self.y[i],

            dtype=torch.float32
            if self.type == "reg"
            else torch.long
        )

        return item

# ================================================================
# LOADERS
# ================================================================

def build_loaders(tok, hp):

    out = {}

    for task in TASK_LIST:

        m = TASKS[task]

        raw = load_dataset(*m["hf"])

        tr = GlueTaskDS(
            raw["train"],
            task,
            tok,
            hp["max_seq_length"]
        )

        ev = GlueTaskDS(
            raw["validation"],
            task,
            tok,
            hp["max_seq_length"]
        )

        out[task] = {

            "train": DataLoader(

                tr,

                batch_size=hp["batch_size"],

                shuffle=True,

                num_workers=hp["num_workers"],

                pin_memory=True,

                drop_last=True
            ),

            "eval": DataLoader(

                ev,

                batch_size=hp["eval_batch_size"],

                shuffle=False,

                num_workers=hp["num_workers"],

                pin_memory=True
            ),
        }

    return out

# ================================================================
# MODEL
# ================================================================

class ClsHead(nn.Module):

    def __init__(self, h, n, dr):

        super().__init__()

        self.do = nn.Dropout(dr)

        self.d = nn.Linear(h, h)

        self.a = nn.Tanh()

        self.o = nn.Linear(h, n)

    def forward(self, x):

        x = self.do(x)

        x = self.a(self.d(x))

        x = self.do(x)

        return self.o(x)

class RegHead(nn.Module):

    def __init__(self, h, dr):

        super().__init__()

        self.do = nn.Dropout(dr)

        self.d = nn.Linear(h, h)

        self.a = nn.Tanh()

        self.o = nn.Linear(h, 1)

    def forward(self, x):

        x = self.do(x)

        x = self.a(self.d(x))

        x = self.do(x)

        return self.o(x).squeeze(-1)

class MTLModel(nn.Module):

    def __init__(self):

        super().__init__()

        cfg = AutoConfig.from_pretrained(
            MODEL_NAME
        )

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME,
            config=cfg
        )

        self.heads = nn.ModuleDict({

            "sst2":
                ClsHead(HIDDEN_SIZE, 2, HP["hidden_dropout"]),

            "qqp":
                ClsHead(HIDDEN_SIZE, 2, HP["hidden_dropout"]),

            "stsb":
                RegHead(HIDDEN_SIZE, HP["hidden_dropout"]),
        })

        self.lcls = nn.CrossEntropyLoss()

        self.lreg = nn.MSELoss()

    def pool(self, **kw):

        o = self.encoder(**kw)

        if getattr(
            o,
            "pooler_output",
            None
        ) is not None:

            return o.pooler_output

        return o.last_hidden_state[:, 0, :]

    def forward(
        self,
        task,
        input_ids,
        attention_mask=None,
        token_type_ids=None,
        labels=None
    ):

        kw = {
            "input_ids": input_ids
        }

        if attention_mask is not None:
            kw["attention_mask"] = attention_mask

        if token_type_ids is not None:
            kw["token_type_ids"] = token_type_ids

        cls = self.pool(**kw)

        z = self.heads[task](cls)

        if labels is None:

            return None, z

        loss = (

            self.lreg(z, labels)

            if task == "stsb"

            else self.lcls(z, labels)
        )

        return loss, z

print("CELL 1 DONE")

Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 47.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.
DEVICE: cuda
CELL 1 DONE


In [ ]:
# ================================================================
# CELL 2: TRAIN + EVAL + CHECKPOINT + CSV + RESUME
# AUTO LOAD LATEST CHECKPOINT
# ================================================================

def move_batch(batch, device):

    return {

        k: v.to(device)

        for k, v in batch.items()

        if isinstance(v, torch.Tensor)
    }

# ================================================================
# PARAM COUNT
# ================================================================

def count_params(model):

    total = sum(
        p.numel()
        for p in model.parameters()
    )

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return int(trainable), int(total)

# ================================================================
# EVALUATE
# ================================================================

@torch.no_grad()
def evaluate(model, loaders, device):

    model.eval()

    results = {}

    losses_all = []

    for task in TASK_LIST:

        preds = []
        labels = []
        losses = []

        for batch in loaders[task]["eval"]:

            b = move_batch(batch, device)

            y = b.pop("labels")

            loss, z = model(

                task,

                b.get("input_ids"),

                b.get("attention_mask"),

                b.get("token_type_ids"),

                labels=y
            )

            losses.append(float(loss.item()))

            if TASKS[task]["type"] == "cls":

                preds.append(
                    z.argmax(-1).cpu().numpy()
                )

            else:

                preds.append(
                    z.squeeze(-1).cpu().numpy()
                )

            labels.append(
                y.cpu().numpy()
            )

        yp = np.concatenate(preds)
        yt = np.concatenate(labels)

        el = float(np.mean(losses))

        losses_all.append(el)

        if TASKS[task]["type"] == "cls":

            results[task] = {

                "accuracy":
                    float(
                        accuracy_score(yt, yp)
                    ),

                "precision":
                    float(
                        precision_score(
                            yt,
                            yp,
                            average="macro",
                            zero_division=0
                        )
                    ),

                "recall":
                    float(
                        recall_score(
                            yt,
                            yp,
                            average="macro",
                            zero_division=0
                        )
                    ),

                "macro_f1":
                    float(
                        f1_score(
                            yt,
                            yp,
                            average="macro",
                            zero_division=0
                        )
                    ),

                "eval_loss": el,
            }

        else:

            if np.std(yp) < 1e-9:

                p = 0.0
                s = 0.0

            else:

                p = float(
                    pearsonr(yp, yt)[0]
                )

                s = float(
                    spearmanr(yp, yt)[0]
                )

            results[task] = {

                "pearson": p,

                "spearman": s,

                "eval_loss": el,
            }

    overall = float(np.mean([

        results["sst2"]["accuracy"],

        results["qqp"]["accuracy"],

        results["stsb"]["pearson"]
    ]))

    return (

        results,

        overall,

        float(np.mean(losses_all))
    )

# ================================================================
# BENCHMARK
# ================================================================

@torch.no_grad()
def benchmark(
    model,
    loaders,
    device,
    max_batches=10
):

    model.eval()

    if torch.cuda.is_available():

        torch.cuda.reset_peak_memory_stats(device)

        torch.cuda.synchronize()

    n = 0

    t0 = time.perf_counter()

    for task in TASK_LIST:

        for i, batch in enumerate(loaders[task]["eval"]):

            if i >= max_batches:
                break

            b = move_batch(batch, device)

            b.pop("labels", None)

            n += int(
                b["input_ids"].shape[0]
            )

            _ = model(

                task,

                b.get("input_ids"),

                b.get("attention_mask"),

                b.get("token_type_ids")
            )

    if torch.cuda.is_available():

        torch.cuda.synchronize()

    elapsed = time.perf_counter() - t0

    peak_vram_mb = (

        float(
            torch.cuda.max_memory_allocated(device)
            / 1024**2
        )

        if torch.cuda.is_available()

        else 0.0
    )

    return {

        "inference_latency_ms":
            (elapsed / max(n, 1)) * 1000.0,

        "throughput_samples_per_sec":
            n / max(elapsed, 1e-9),

        "peak_vram_mb":
            peak_vram_mb,

        "benchmark_samples":
            n,
    }

# ================================================================
# CHECKPOINT
# ================================================================

def save_ckpt(path, state):

    Path(path).parent.mkdir(
        parents=True,
        exist_ok=True
    )

    torch.save(state, path)

def rotate_ckpts(ckpt_dir, keep_n):

    ckpts = sorted(

        Path(ckpt_dir).glob(
            "ckpt_epoch*.pt"
        ),

        key=lambda p: int(
            p.stem.replace(
                "ckpt_epoch",
                ""
            )
        )
    )

    for ck in ckpts[:-keep_n]:

        try:
            ck.unlink()
        except:
            pass

# ================================================================
# TRAIN
# ================================================================

def train():

    out_dir = (
        Path(OUTPUT_ROOT)
        / EXP_NAME
    )

    out_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    ckpt_dir = (
        out_dir
        / "checkpoints"
    )

    ckpt_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    logger = make_logger(
        EXP_NAME,
        str(out_dir / "train.log")
    )

    tok = AutoTokenizer.from_pretrained(
        MODEL_NAME
    )

    loaders = build_loaders(
        tok,
        HP
    )

    model = MTLModel().to(DEVICE)

    trainable, total = count_params(model)

    print("\nTRAINABLE:", trainable)

    print("TOTAL:", total)

    optimizer = AdamW(

        model.parameters(),

        lr=HP["learning_rate"],

        weight_decay=HP["weight_decay"]
    )

    nsteps = max(

        len(loaders[t]["train"])

        for t in TASK_LIST
    )

    total_steps = (
        nsteps
        * HP["max_epochs"]
    )

    warm = int(
        HP["warmup_ratio"]
        * total_steps
    )

    scheduler = get_linear_schedule_with_warmup(

        optimizer,

        warm,

        total_steps
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

    hist = HistoryWriter(out_dir)

    best = -999999

    patience = 0

    start_epoch = 1

    # ============================================================
    # AUTO LOAD LATEST CHECKPOINT
    # ============================================================

    ckpts = sorted(

        Path(ckpt_dir).glob(
            "ckpt_epoch*.pt"
        ),

        key=lambda p: int(
            p.stem.replace(
                "ckpt_epoch",
                ""
            )
        )
    )

    latest = ckpts[-1] if len(ckpts) > 0 else None

    if latest is not None:

        logger.info(
            f"RESUME FROM {latest}"
        )

        st = torch.load(

            latest,

            map_location=DEVICE
        )

        model.load_state_dict(
            st["model"]
        )

        optimizer.load_state_dict(
            st["optimizer"]
        )

        scheduler.load_state_dict(
            st["scheduler"]
        )

        start_epoch = (
            int(st["epoch"]) + 1
        )

        best = st["best"]

        patience = st["pat"]

        hist.records = st["history"]

        logger.info(
            f"START EPOCH = {start_epoch}"
        )

    else:

        logger.info(
            "NO CHECKPOINT FOUND"
        )

    best_record = {}

    total_train_start = time.perf_counter()

    # ============================================================
    # TRAIN LOOP
    # ============================================================

    for epoch in range(

        start_epoch,

        HP["max_epochs"] + 1
    ):

        logger.info(
            f"START TRAIN EPOCH {epoch}"
        )

        model.train()

        train_loss = 0.0

        nb = 0

        t0 = time.perf_counter()

        train_iters = {

            t: iter(loaders[t]["train"])

            for t in TASK_LIST
        }

        for step in range(nsteps):

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.amp.autocast(
                "cuda",
                enabled=USE_AMP
            ):

                losses = []

                for task in TASK_LIST:

                    try:

                        batch = next(
                            train_iters[task]
                        )

                    except StopIteration:

                        train_iters[task] = iter(
                            loaders[task]["train"]
                        )

                        batch = next(
                            train_iters[task]
                        )

                    b = move_batch(
                        batch,
                        DEVICE
                    )

                    y = b.pop("labels")

                    loss, _ = model(

                        task,

                        b.get("input_ids"),

                        b.get("attention_mask"),

                        b.get("token_type_ids"),

                        labels=y
                    )

                    losses.append(loss)

                total_loss = sum(losses) / K

            scaler.scale(total_loss).backward()

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(

                model.parameters(),

                HP["grad_clip"]
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            train_loss += float(
                total_loss.item()
            )

            nb += 1

        epoch_time = (
            time.perf_counter()
            - t0
        )

        train_loss /= max(nb, 1)

        # ========================================================
        # EVAL
        # ========================================================

        results, overall, eval_loss = evaluate(

            model,

            loaders,

            DEVICE
        )

        bench = benchmark(

            model,

            loaders,

            DEVICE,

            max_batches=10
        )

        # ========================================================
        # RECORD
        # ========================================================

        record = {

            "epoch": epoch,

            "train_loss": train_loss,

            "eval_loss": eval_loss,

            "sst2_accuracy":
                results["sst2"]["accuracy"],

            "sst2_macro_f1":
                results["sst2"]["macro_f1"],

            "sst2_precision":
                results["sst2"]["precision"],

            "sst2_recall":
                results["sst2"]["recall"],

            "qqp_accuracy":
                results["qqp"]["accuracy"],

            "qqp_macro_f1":
                results["qqp"]["macro_f1"],

            "qqp_precision":
                results["qqp"]["precision"],

            "qqp_recall":
                results["qqp"]["recall"],

            "stsb_pearson":
                results["stsb"]["pearson"],

            "stsb_spearman":
                results["stsb"]["spearman"],

            "overall_score":
                overall,

            "time_per_epoch":
                epoch_time,

            "peak_vram_mb":
                bench["peak_vram_mb"],

            "throughput_samples_per_sec":
                bench["throughput_samples_per_sec"],

            "inference_latency_ms":
                bench["inference_latency_ms"],
        }

        hist.append(record)

        logger.info(record)

        # ========================================================
        # SAVE CKPT
        # ========================================================

        save_ckpt(

            ckpt_dir / f"ckpt_epoch{epoch:02d}.pt",

            {

                "model":
                    model.state_dict(),

                "optimizer":
                    optimizer.state_dict(),

                "scheduler":
                    scheduler.state_dict(),

                "epoch":
                    epoch,

                "best":
                    best,

                "pat":
                    patience,

                "history":
                    hist.records,
            }
        )

        rotate_ckpts(
            ckpt_dir,
            HP["keep_n_ckpts"]
        )

        # ========================================================
        # BEST MODEL
        # ========================================================

        if overall > best:

            best = overall

            patience = 0

            best_record = dict(record)

            save_ckpt(

                out_dir / "best_model.pt",

                {

                    "model":
                        model.state_dict(),

                    "epoch":
                        epoch,

                    "best":
                        best,
                }
            )

            logger.info(
                f"NEW BEST {best:.4f}"
            )

        else:

            patience += 1

            logger.info(
                f"NO IMPROVEMENT "
                f"{patience}/{HP['patience']}"
            )

        # ========================================================
        # EARLY STOPPING
        # ========================================================

        if patience >= HP["patience"]:

            logger.info(
                "EARLY STOPPING"
            )

            break

    # ============================================================
    # FINAL MODEL
    # ============================================================

    save_ckpt(

        out_dir / "final_model.pt",

        {

            "model":
                model.state_dict(),

            "epoch":
                epoch,
        }
    )

    total_train_time = (

        time.perf_counter()

        - total_train_start
    )

    # ============================================================
    # SUMMARY
    # ============================================================

    summary = {

        "model": MODEL_NAME,

        "best_epoch":
            best_record.get("epoch"),

        "best_overall_score":
            best,

        "best_sst2_accuracy":
            best_record.get("sst2_accuracy"),

        "best_sst2_macro_f1":
            best_record.get("sst2_macro_f1"),

        "best_qqp_accuracy":
            best_record.get("qqp_accuracy"),

        "best_qqp_macro_f1":
            best_record.get("qqp_macro_f1"),

        "best_stsb_pearson":
            best_record.get("stsb_pearson"),

        "best_stsb_spearman":
            best_record.get("stsb_spearman"),

        "peak_vram_mb":
            best_record.get("peak_vram_mb"),

        "throughput_samples_per_sec":
            best_record.get(
                "throughput_samples_per_sec"
            ),

        "inference_latency_ms":
            best_record.get(
                "inference_latency_ms"
            ),

        "trainable_params":
            trainable,

        "total_params":
            total,

        "total_train_time_s":
            total_train_time,
    }

    with open(

        out_dir / "summary.json",

        "w",

        encoding="utf-8"
    ) as f:

        json.dump(
            summary,
            f,
            indent=2,
            default=str
        )

    with open(

        out_dir / "summary.csv",

        "w",

        newline="",

        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=summary.keys()
        )

        writer.writeheader()

        writer.writerow(summary)

    print("\n")

    print("=" * 60)

    print("TRAINING FINISHED")

    print("=" * 60)

    print(summary)

    return summary

# ================================================================
# START TRAIN
# ================================================================

train()

print("\nCELL 2 DONE")

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(



TRAINABLE: 33805445
TOTAL: 33805445
[2026-05-18 17:27:53] [INFO] RESUME FROM /content/drive/MyDrive/MINILM_MTL_RESULTS/MiniLM-MTL-FFT/checkpoints/ckpt_epoch04.pt
[2026-05-18 17:28:01] [INFO] START EPOCH = 5
[2026-05-18 17:28:01] [INFO] START TRAIN EPOCH 5
[2026-05-18 19:04:42] [INFO] {'epoch': 5, 'train_loss': 0.0685502010060756, 'eval_loss': 0.5239422662357001, 'sst2_accuracy': 0.9002293577981652, 'sst2_macro_f1': 0.900086264231952, 'sst2_precision': 0.901021792253168, 'sst2_recall': 0.8998379220341837, 'qqp_accuracy': 0.9045263418253772, 'qqp_macro_f1': 0.8977695887134924, 'qqp_precision': 0.8963753488691444, 'qqp_recall': 0.8992574966174086, 'stsb_pearson': 0.8600237369537354, 'stsb_spearman': 0.8798609960990809, 'overall_score': 0.8882598121924259, 'time_per_epoch': 5582.006695368, 'peak_vram_mb': 1480.44580078125, 'throughput_samples_per_sec': 197.99033690959524, 'inference_latency_ms': 5.050751544791865}
[2026-05-18 19:04:44] [INFO] NO IMPROVEMENT 2/3


TRAINING FINISHED
{'model